# 1. Project Context
## Coordinated Sampling for Relational Datasets

## Objective
This notebook implements a **consistent sampling strategy** across multiple related datasets.

We are working with three datasets:
- Crashes (parent table)
- Vehicles (child table)
- People (child table)

## Problem
Sampling each dataset independently breaks relationships between tables.

## Solution
We:
1. Sample from the parent table (crashes)
2. Extract the primary keys (CRASH_RECORD_ID)
3. Filter child tables (Vehiclesm, People) using those keys

This ensures:
- Referential integrity
- Consistent subset across all datasets

## 2. Imports & Config

## **Original Data Source**

https://catalog.data.gov/dataset/traffic-crashes-crashes 

https://catalog.data.gov/dataset/traffic-crashes-vehicles 

https://catalog.data.gov/dataset/traffic-crashes-people

In [ ]:
import pandas as pd
import os

# File paths 
crashes_path = "original_crashes_data\Traffic_Crashes_-_Crashes.csv"
vehicles_path = "original_crashes_data\Traffic_Crashes_-_Vehicles.csv"
people_path = "original_crashes_data\Traffic_Crashes_-_People.csv"

# Output directory ----> storing raw data to the Bronze layer
output_dir = "bronze"
os.makedirs(output_dir, exist_ok=True)

# Sample size
s_size = 250_000
r_state = 42

## 3. **Load Crashes Dataset (Parent Table)**

In [24]:
crashes = pd.read_csv(crashes_path)

print(crashes.shape)
crashes.head()

(1049082, 48)


,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION
0,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...,NaN,01/14/2025 12:25:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,SIDESWIPE SAME DIRECTION,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,0.0,2.0,0.0,12,3,1,41.997808,-87.655770,POINT (-87.655770494712 41.997807727633)
1,027b0b4c21460d3441fd83929abb9673c6fc0c7d575675...,NaN,05/23/2025 09:30:00 AM,30,STOP SIGN/FLASHER,UNKNOWN,UNKNOWN,DAYLIGHT,TURNING,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,0.0,2.0,0.0,9,6,5,41.946529,-87.688106,POINT (-87.688106391039 41.946529480518)
2,04d91dffc94f677358ca47056921ba5c4224320df27ed4...,Y,04/05/2025 08:00:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,UNKNOWN,PARKED MOTOR VEHICLE,NOT DIVIDED,...,0.0,0.0,1.0,0.0,20,7,4,41.899325,-87.715074,POINT (-87.715074373867 41.899324573751)
3,0b5603954d84b7341c7cad4f570ea039e85919f3750ccb...,NaN,05/23/2025 09:15:00 AM,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,PEDALCYCLIST,NOT DIVIDED,...,1.0,0.0,2.0,0.0,9,6,5,41.902793,-87.699412,POINT (-87.699412181285 41.902792968177)
4,00bce77960c2faa2a8782a8cac1d6e5715802d6c072a08...,NaN,01/14/2025 08:00:00 AM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,REAR END,FOUR WAY,...,0.0,0.0,2.0,0.0,8,3,1,41.691207,-87.720555,POINT (-87.720554863466 41.691206664451)


In [25]:
crashes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1049082 entries, 0 to 1049081
Data columns (total 48 columns):
 #   Column                         Non-Null Count    Dtype  
---  ------                         --------------    -----  
 0   CRASH_RECORD_ID                1049082 non-null  object 
 1   CRASH_DATE_EST_I               75955 non-null    object 
 2   CRASH_DATE                     1049082 non-null  object 
 3   POSTED_SPEED_LIMIT             1049082 non-null  int64  
 4   TRAFFIC_CONTROL_DEVICE         1049082 non-null  object 
 5   DEVICE_CONDITION               1049082 non-null  object 
 6   WEATHER_CONDITION              1049082 non-null  object 
 7   LIGHTING_CONDITION             1049082 non-null  object 
 8   FIRST_CRASH_TYPE               1049082 non-null  object 
 9   TRAFFICWAY_TYPE                1049082 non-null  object 
 10  LANE_CNT                       199043 non-null   float64
 11  ALIGNMENT                      1049082 non-null  object 
 12  ROADWAY_SURFAC

## 4. Sample from Crashes

In [26]:
crashes_sample = crashes.sample(s_size, random_state= r_state) 
print(crashes_sample.shape)

(250000, 48)


## 5. Extract Keys

In [27]:
crash_ids = crashes_sample['CRASH_RECORD_ID'].unique()

crashes_ids_set = set(crash_ids) 

print(f"Unique crash IDs:\n{len(crashes_ids_set)}")

Unique crash IDs:
250000


## 6. Save Crashes Sample (Raw/Bronze Layer)

In [28]:
crashes_sample.to_csv(f"{output_dir}/crashes_sample.csv", index= False)


## 7. **Filter *Vehicles* using Chunking** 
because the original file size is large

In [30]:
chunks = []

for chunk in pd.read_csv(vehicles_path, chunksize= 100_000):
    filtered = chunk[ chunk['CRASH_RECORD_ID'].
                        isin(crashes_ids_set) ] 
    chunks.append(filtered)

vehicles_sample = pd.concat(chunks, ignore_index= True)

print(vehicles_sample.shape)
vehicles_sample.head()

C:\Users\hp\AppData\Local\Temp\ipykernel_24404\3039945259.py:3: DtypeWarning: Columns (20,39,40,41,47,48,49,52,54,57,58,59,60,70) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(vehicles_path, chunksize= 100_000):
C:\Users\hp\AppData\Local\Temp\ipykernel_24404\3039945259.py:3: DtypeWarning: Columns (39,40,41,47,48,49,57,58,60) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(vehicles_path, chunksize= 100_000):
C:\Users\hp\AppData\Local\Temp\ipykernel_24404\3039945259.py:3: DtypeWarning: Columns (39,40,41,47,48,49,54,57,58,59,60,70) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(vehicles_path, chunksize= 100_000):
C:\Users\hp\AppData\Local\Temp\ipykernel_24404\3039945259.py:3: DtypeWarning: Columns (20,39,40,41,47,48,49,57,58,60) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(v

(509881, 71)


,CRASH_UNIT_ID,CRASH_RECORD_ID,CRASH_DATE,UNIT_NO,UNIT_TYPE,NUM_PASSENGERS,VEHICLE_ID,CMRC_VEH_I,MAKE,MODEL,...,TRAILER1_LENGTH,TRAILER2_LENGTH,TOTAL_VEHICLE_LENGTH,AXLE_CNT,VEHICLE_CONFIG,CARGO_BODY_TYPE,LOAD_TYPE,HAZMAT_OUT_OF_SERVICE_I,MCS_OUT_OF_SERVICE_I,HAZMAT_CLASS
0,2291788,5cbc1f459ffeb652279728a8f225bdc7177ab4361f499d...,04/26/2026 01:54:00 AM,1,DRIVER,4.0,2185715.0,NaN,LINCOLN,NAVIGATOR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2291789,5cbc1f459ffeb652279728a8f225bdc7177ab4361f499d...,04/26/2026 01:54:00 AM,2,PARKED,NaN,2185719.0,NaN,TOYOTA,SIENNA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2291781,b75606a44b0244f91e54ebb3e1fc238446b57ebd25a46f...,04/26/2026 01:45:00 AM,1,DRIVER,NaN,2185712.0,NaN,UNKNOWN,OTHER (EXPLAIN IN NARRATIVE),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2291782,b75606a44b0244f91e54ebb3e1fc238446b57ebd25a46f...,04/26/2026 01:45:00 AM,2,DRIVER,NaN,2185721.0,NaN,NISSAN,KICKS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2291723,0c96376af0237ab7f392c1e62c12bd1500595707a6cb7a...,04/25/2026 10:05:00 PM,1,DRIVER,NaN,2185650.0,NaN,UNKNOWN,OTHER (EXPLAIN IN NARRATIVE),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
vehicles_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 509881 entries, 0 to 509880
Data columns (total 71 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   CRASH_UNIT_ID             509881 non-null  int64  
 1   CRASH_RECORD_ID           509881 non-null  object 
 2   CRASH_DATE                509881 non-null  object 
 3   UNIT_NO                   509881 non-null  object 
 4   UNIT_TYPE                 509312 non-null  object 
 5   NUM_PASSENGERS            75236 non-null   float64
 6   VEHICLE_ID                497844 non-null  float64
 7   CMRC_VEH_I                9598 non-null    object 
 8   MAKE                      497843 non-null  object 
 9   MODEL                     497817 non-null  object 
 10  LIC_PLATE_STATE           452596 non-null  object 
 11  VEHICLE_YEAR              419964 non-null  float64
 12  VEHICLE_DEFECT            497844 non-null  object 
 13  VEHICLE_TYPE              497844 non-null  o

### Save Vehicles Sample / new file

In [32]:
vehicles_sample.to_csv(f"{output_dir}/vehicles_sample.csv", index= False)


## 8. **Filter *People* using Chunking**

In [33]:
chunks = []

for chunk in pd.read_csv(people_path, chunksize= 100_000):
    filtered = chunk[ chunk['CRASH_RECORD_ID'].
                        isin(crashes_ids_set) ] 
    chunks.append(filtered)

people_sample = pd.concat(chunks, ignore_index= True)

print(people_sample.shape)
people_sample.head()

C:\Users\hp\AppData\Local\Temp\ipykernel_24404\2913697839.py:3: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(people_path, chunksize= 100_000):
C:\Users\hp\AppData\Local\Temp\ipykernel_24404\2913697839.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(people_path, chunksize= 100_000):
C:\Users\hp\AppData\Local\Temp\ipykernel_24404\2913697839.py:3: DtypeWarning: Columns (8,28) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(people_path, chunksize= 100_000):
C:\Users\hp\AppData\Local\Temp\ipykernel_24404\2913697839.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(people_path, chunksize= 100_000):
C:\Users\hp\AppData\Local\Temp\ipykernel_24404\2913697839.py:3: DtypeWarning: Columns (19) have 

(549180, 29)


,PERSON_ID,PERSON_TYPE,CRASH_RECORD_ID,VEHICLE_ID,CRASH_DATE,SEAT_NO,CITY,STATE,ZIPCODE,SEX,...,EMS_RUN_NO,DRIVER_ACTION,DRIVER_VISION,PHYSICAL_CONDITION,PEDPEDAL_ACTION,PEDPEDAL_VISIBILITY,PEDPEDAL_LOCATION,BAC_RESULT,BAC_RESULT VALUE,CELL_PHONE_USE
0,O10018,DRIVER,71162af7bf22799b776547132ebf134b5b438dcf3dac6b...,9579.0,11/01/2015 05:00:00 AM,NaN,NaN,NaN,NaN,X,...,NaN,IMPROPER BACKING,UNKNOWN,UNKNOWN,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
1,O10041,DRIVER,dd1bce4bd6d0be4c247714dcabab44e6563c62b913229b...,9601.0,11/01/2015 11:00:00 AM,NaN,NaN,NaN,NaN,X,...,NaN,UNKNOWN,UNKNOWN,UNKNOWN,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
2,O10062,DRIVER,4bd2ee6bb306902b99a9c2ae55cf4fcffec00879e39759...,9621.0,11/01/2015 12:30:00 PM,NaN,NaN,NaN,NaN,X,...,NaN,UNKNOWN,UNKNOWN,UNKNOWN,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
3,O877654,DRIVER,e9146986f4b0884d00ff3a54da5249263b4b36c15d01ce...,832624.0,04/30/2020 03:05:00 PM,NaN,CHICAGO,IL,60620,M,...,NaN,UNKNOWN,UNKNOWN,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN
4,O879085,DRIVER,f15ccbd94a8e29ce8424882ce93061d4e1d0deb214acfe...,833984.0,05/03/2020 10:30:00 PM,NaN,CALUMENT CITY,NaN,NaN,M,...,NaN,UNKNOWN,UNKNOWN,NORMAL,NaN,NaN,NaN,TEST NOT OFFERED,NaN,NaN


In [34]:
people_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 549180 entries, 0 to 549179
Data columns (total 29 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   PERSON_ID              549180 non-null  object 
 1   PERSON_TYPE            549180 non-null  object 
 2   CRASH_RECORD_ID        549180 non-null  object 
 3   VEHICLE_ID             537752 non-null  float64
 4   CRASH_DATE             549180 non-null  object 
 5   SEAT_NO                110681 non-null  float64
 6   CITY                   399769 non-null  object 
 7   STATE                  405895 non-null  object 
 8   ZIPCODE                369367 non-null  object 
 9   SEX                    539482 non-null  object 
 10  AGE                    389998 non-null  float64
 11  DRIVERS_LICENSE_STATE  322376 non-null  object 
 12  DRIVERS_LICENSE_CLASS  266187 non-null  object 
 13  SAFETY_EQUIPMENT       547720 non-null  object 
 14  AIRBAG_DEPLOYED        538179 non-nu

### Save People Sample

In [35]:
people_sample.to_csv(f"{output_dir}/people_sample.csv", index= False)


## 9. Validation  

In [36]:
vehicles_valid = vehicles_sample["CRASH_RECORD_ID"]\
                            .isin(crashes_ids_set)\
                            .all()

people_valid = people_sample["CRASH_RECORD_ID"]\
                            .isin(crashes_ids_set)\
                            .all()

print("Vehicles valid:", vehicles_valid)
print("People valid:", people_valid)

Vehicles valid: True
People valid: True


## 10. Summary

In [37]:
print("Final Dataset Sizes:")
print(f"Crashes: {crashes_sample.shape}")
print(f"Vehicles: {vehicles_sample.shape}")
print(f"People: {people_sample.shape}")

Final Dataset Sizes:
Crashes: (250000, 48)
Vehicles: (509881, 71)
People: (549180, 29)


## Coordinated Sampling Strategy

### Why this approach?
Sampling child tables independently would break relationships between datasets.

### What we did:
1. Selected a random sample from the parent table (crashes)
2. Extracted unique CRASH_RECORD_ID values
3. Filtered related datasets (vehicles, people) using these keys

### Benefits:
- Maintains relational integrity
- Ensures consistency across datasets
- Reduces data size for faster processing
- Suitable for development and testing pipelines

### Output:
We generated three sampled datasets:
- crashes_sample.csv
- vehicles_sample.csv
- people_sample.csv

These files serve as the **raw layer (Bronze)** for the pipeline.

### resaving the files converting csv to pipe separated

In [2]:
import pandas as pd

# Run this once per file — no transformation, just re-saving
for name in ['crashes', 'people', 'vehicles']:
    df = pd.read_csv(rf'raw_datasets\{name}.csv', low_memory=False)
    df.to_csv(rf'raw_datasets\{name}_pipe.csv', sep='|', index=False)
    print(f"{name}: {len(df)} rows saved")

crashes: 250000 rows saved
people: 549180 rows saved
vehicles: 509881 rows saved


## Loading the file vehicles directly to bronze layer (in SQL Server) 
duo to encoding problems

In [9]:
%pip install pyodbc sqlalchemy

Note: you may need to restart the kernel to use updated packages.


## **Loading the vehicles data directly to SQL Server Bronze schema**
due to encoding and raw data problems

In [11]:
import pandas as pd
import pyodbc

CHUNKSIZE = 5000  # you can try 5k–20k



conn = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=localhost;'
    'DATABASE=crashes_DWH;'
    'Trusted_Connection=yes;'
)
cursor = conn.cursor()
cursor.fast_executemany = True

# Truncate once
cursor.execute("TRUNCATE TABLE bronze.vehicles")
conn.commit()

print("Table truncated")

# Start reading in chunks
for i, chunk in enumerate(pd.read_csv(
        r'raw_datasets\vehicles.csv',
        chunksize=CHUNKSIZE,
        encoding='utf-8',
        dtype=str   # 🔥 THIS LINE FIXES EVERYTHING
    )):

    print(f"Processing chunk {i+1}...")

    # Clean quotes
    for col in chunk.select_dtypes(include='object').columns:
        chunk[col] = chunk[col].str.replace('"', '', regex=False)

    # Replace NaN with None
    chunk = chunk.where(pd.notna(chunk), None)

    # Prepare insert
    cols = ', '.join(chunk.columns)
    params = ', '.join(['?' for _ in chunk.columns])
    sql = f"INSERT INTO bronze.vehicles ({cols}) VALUES ({params})"

    rows = list(chunk.itertuples(index=False, name=None))

    cursor.executemany(sql, rows)

    print(f"Inserted {len(rows)} rows")

conn.commit()

cursor.close()
conn.close()

print("✅ DONE")

Table truncated
Processing chunk 1...
Inserted 5000 rows
Processing chunk 2...
Inserted 5000 rows
Processing chunk 3...
Inserted 5000 rows
Processing chunk 4...
Inserted 5000 rows
Processing chunk 5...
Inserted 5000 rows
Processing chunk 6...
Inserted 5000 rows
Processing chunk 7...
Inserted 5000 rows
Processing chunk 8...
Inserted 5000 rows
Processing chunk 9...
Inserted 5000 rows
Processing chunk 10...
Inserted 5000 rows
Processing chunk 11...
Inserted 5000 rows
Processing chunk 12...
Inserted 5000 rows
Processing chunk 13...
Inserted 5000 rows
Processing chunk 14...
Inserted 5000 rows
Processing chunk 15...
Inserted 5000 rows
Processing chunk 16...
Inserted 5000 rows
Processing chunk 17...
Inserted 5000 rows
Processing chunk 18...
Inserted 5000 rows
Processing chunk 19...
Inserted 5000 rows
Processing chunk 20...
Inserted 5000 rows
Processing chunk 21...
Inserted 5000 rows
Processing chunk 22...
Inserted 5000 rows
Processing chunk 23...
Inserted 5000 rows
Processing chunk 24...
Inse

### resaving only the vehicles file due to ecoding problems

In [ ]:
# import pandas as pd
# import csv

# # df = pd.read_csv(r'raw_datasets\vehicles.csv', low_memory=False, encoding='utf-8')

# # Strip double quotes from all string columns
# # Bronze still gets the real value — the quotes are noise, not data
# for col in df.select_dtypes(include='object').columns:
#     df[col] = df[col].str.replace('"', '', regex=False)

# df.to_csv(
#     r'raw_datasets\vehicles_pipe.csv',
#     sep='|',
#     index=False,
#     encoding='utf-8-sig',
#     quoting=csv.QUOTE_NONE,   # ← Never wrap fields in quotes
#     escapechar='\\'           # ← Required when QUOTE_NONE is set
# )

# print(f"vehicles: {len(df)} rows saved")

# # Verify the problem rows are now clean
# for r in [1753, 1961, 3380]:
#     row = df.iloc[r - 2]
#     print(f"Row {r} MAKE=[{row['MAKE']}] MODEL=[{row['MODEL']}]")

vehicles: 509881 rows saved
Row 1753 MAKE=[PETERBILT MOTORS COMPANY (DIVISION OF PACCAR, INC.)] MODEL=[OTHER (EXPLAIN IN NARRATIVE)]
Row 1961 MAKE=[PREVOST CAR / BUS (SAINTE CLAIRE, QUEBEC)] MODEL=[PREVOST CAR, (PREVOST BUS; SAINTE CLAIRE, QUEBEC]
Row 3380 MAKE=[PREVOST CAR / BUS (SAINTE CLAIRE, QUEBEC)] MODEL=[PREVOST CAR, (PREVOST BUS; SAINTE CLAIRE, QUEBEC]


In [ ]:
# # import pandas as pd

# # df = pd.read_csv(r'raw_datasets\vehicles.csv', low_memory=False)

# # Check the exact failing rows
# failing_rows = [1753, 1961, 3380, 4395, 6262, 6309, 7136, 7539, 7562, 8080, 8428]

# # pandas is 0-indexed, BULK INSERT row 1753 = index 1752 (subtract header)
# for r in failing_rows:
#     row = df.iloc[r - 2]  # -1 for header, -1 for 0-index
#     print(f"\nRow {r}:")
#     print(f"  MAKE  = [{repr(row['MAKE'])}]")
#     print(f"  MODEL = [{repr(row['MODEL'])}]")

# # Also verify the pipe file was actually updated
# import os
# path = r'raw_datasets\vehicles_pipe.csv'
# print(f"\nFile last modified: {pd.Timestamp(os.path.getmtime(path), unit='s')}")
# print(f"File size: {os.path.getsize(path):,} bytes")

# # Check if pipe exists in MAKE or MODEL
# pipe_in_make  = df['MAKE'].str.contains(r'\|', na=False).sum()
# pipe_in_model = df['MODEL'].str.contains(r'\|', na=False).sum()
# print(f"\nPipe chars in MAKE:  {pipe_in_make}")
# print(f"Pipe chars in MODEL: {pipe_in_model}")


Row 1753:
  MAKE  = ['PETERBILT MOTORS COMPANY (DIVISION OF PACCAR, INC.)']
  MODEL = ['OTHER (EXPLAIN IN NARRATIVE)']

Row 1961:
  MAKE  = ['PREVOST CAR / BUS (SAINTE CLAIRE, QUEBEC)']
  MODEL = ['PREVOST CAR, ("PREVOST BUS;" SAINTE CLAIRE, QUEBEC']

Row 3380:
  MAKE  = ['PREVOST CAR / BUS (SAINTE CLAIRE, QUEBEC)']
  MODEL = ['PREVOST CAR, ("PREVOST BUS;" SAINTE CLAIRE, QUEBEC']

Row 4395:
  MAKE  = ['PETERBILT MOTORS COMPANY (DIVISION OF PACCAR, INC.)']
  MODEL = ['OTHER (EXPLAIN IN NARRATIVE)']

Row 6262:
  MAKE  = ['PREVOST CAR / BUS (SAINTE CLAIRE, QUEBEC)']
  MODEL = ['PREVOST CAR, ("PREVOST BUS;" SAINTE CLAIRE, QUEBEC']

Row 6309:
  MAKE  = ['PREVOST CAR / BUS (SAINTE CLAIRE, QUEBEC)']
  MODEL = ['PREVOST CAR, ("PREVOST BUS;" SAINTE CLAIRE, QUEBEC']

Row 7136:
  MAKE  = ['PETERBILT MOTORS COMPANY (DIVISION OF PACCAR, INC.)']
  MODEL = ['PETERBILT MOTORS CO., DIVISION PACCAR, INC.']

Row 7539:
  MAKE  = ['PREVOST CAR / BUS (SAINTE CLAIRE, QUEBEC)']
  MODEL = ['PREVOST CAR, ("PRE